# Heston implied-vol smile

Price calls for $K = 70, 72, \dots, 130$ at $T = 1$ with `heston.pricer.call`, invert to Black-Scholes implied vols, and plot IV vs $K$.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so `import heston` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from heston.params import HestonParams
from heston.pricer import call
from heston.bs import implied_vol

%matplotlib inline

## Helpers

In [ ]:
STRIKES = np.arange(70, 131, 2, dtype=float)

# fixed series colors, assigned in order (never cycled)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]


def heston_smile(params, S0=100.0, r=0.03, T=1.0, q=0.0, strikes=STRIKES):
    """Heston call prices and BS implied vols across strikes."""
    rows = []
    for K in strikes:
        price = call(S0, K, r, T, params, q)
        rows.append({"K": K, "price": price, "iv": implied_vol(price, S0, K, r, T, q)})
    return pd.DataFrame(rows)


def plot_smile(smiles, title="Heston implied vol smile", S0=None):
    """smiles: a DataFrame from heston_smile, or a dict {label: DataFrame}."""
    if isinstance(smiles, pd.DataFrame):
        smiles = {None: smiles}
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for color, (label, df) in zip(COLORS, smiles.items()):
        ax.plot(df["K"], 100 * df["iv"], color=color, lw=2,
                marker="o", ms=4, label=label)
    if S0 is not None:
        ax.axvline(S0, color="0.6", lw=1, ls="--")
    ax.set_xlabel("Strike K")
    ax.set_ylabel("Implied vol (%)")
    ax.set_title(title, loc="left")
    ax.grid(True, color="0.9", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if len(smiles) > 1:
        ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.show()
    return ax

## Parameters (change these)

In [ ]:
S0, r, T, q = 100.0, 0.03, 1.0, 0.0

params = HestonParams(v0=0.04, kappa=2.0, vbar=0.04, eta=0.5, rho=-0.7)
print(params, "| Feller ok:", params.feller_ok())

In [ ]:
smile = heston_smile(params, S0, r, T, q)
plot_smile(smile, S0=S0)
smile.head()

## Compare: vary one parameter (change these)

In [ ]:
base = dict(v0=0.04, kappa=2.0, vbar=0.04, eta=0.5)
rhos = [-0.7, 0.0, 0.7]

smiles = {f"rho = {rho}": heston_smile(HestonParams(**base, rho=rho), S0, r, T, q) for rho in rhos}
plot_smile(smiles, title="Effect of rho on the smile", S0=S0);

In [ ]:
base = dict(v0=0.04, kappa=2.0, vbar=0.04, rho=-0.7)
etas = [0.1, 0.5, 1.0]

smiles = {f"eta = {eta}": heston_smile(HestonParams(**base, eta=eta), S0, r, T, q) for eta in etas}
plot_smile(smiles, title="Effect of eta on the smile", S0=S0);